# Spark transformations
## What do we mean by transformations?
![transformations](images/transformations.png)
- In Spark we read the data from a data source and create one of the two things. 
    - DataFrames
        - The DataFrame is the programmatic interface for your data
        - Transformation : Here the programmatic approach is implemented when it comes to transformations
    - Database table
        - The Database table is the sql interface for your data.
        - Transformation : Here the sql approach is implemented when it comes to transformations
- Both the database tables and the dataframes are the same but two different interfaces.
-  Transformation is nothing but:
    - Combining DataFrames
    - Aggregating and Summarizing 
    - Applying functions and built-in transformations
    - Using built-in and column-level functions
    - Creating and using UDFs
    - Creating column expressions
    - Referencing rows/columns
### Working with DataFrame rows:
- Spark dataframe is a dataset of rows.
- Each row in the dataFrame is a single record represented by an object of type row.
- Most of the time we do not directly work with the entire row.
- However there are three specific senarios where we have to directly work with the row object.
    - Manually creating rows and dataFrame.
    - Collecting DataFrame rows to the driver.
    - Work with an individual row in spark transformations.
#### Example 1: 
- Here in this example you will see where creating a dataFrame manually on the fly helps in unit testing of the functions and method that you create
- Why creating dataFrame helps manually instead of importing sample data from a csv file?
    - Everytime we have to test a function or a method its not possible to read the csv file and bring in some sample data to create a dataFrame because it will make testing the application significantly slower due to un-necessary I/O overhead.
- In this example I have written a function that converts dataType from string to date type given the column name in a dataFrame
- I have also written a test case using python's built in unit test tool to simulate real world senario to the best of my ability.
#### Here is the final code 
#### dataframe_transformations.py
```python
from pyspark.sql.functions import to_date
class DataFrameTransformations:
    def __init__(self,spark):
        self.spark_object = spark
    def count_by_country(self,spark_df):
        intermediate_result_df = (
                spark_df
                .where("CallType is not null")
                .select("CallType","Zipcode")
                .groupby("CallType","Zipcode")
            )
        row_count = intermediate_result_df.count()
        result_df = row_count.orderBy("count",ascending=False)
        return result_df
    
    """This methods onverts the column with dates in string datatype to date datatype"""
    def convert_to_date_type(self, spark_df, date_format, col_name):
        """Converts a string column to date using the given format."""
        return spark_df.withColumn(col_name, to_date(col_name, date_format))
```
#### unit_test.py
```python
import unittest
import os 
import sys
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(os.path.dirname(CURRENT_DIR))  # one level higher
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from pyspark.sql import SparkSession, Row
from SparkDFTransformations.transformations.dataframe_transformations import DataFrameTransformations
from datetime import date


class TestDataFrameTransformations(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = (
            SparkSession
            .builder
            .appName("PySparkUnitTest")
            .master("local[2]")
            .getOrCreate()
        )
        cls.transformer = DataFrameTransformations(cls.spark)

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()

    def test_convert_to_date_type(self):
        # Sample test DataFrame
        data = [
            Row(id="1", EventDate="3/11/2025"),
            Row(id="2", EventDate="4/11/2025")
        ]

        schema = "id STRING, EventDate STRING"
        df = self.spark.createDataFrame(data, schema)

        # Apply transformation
        result_df = self.transformer.convert_to_date_type(df, "d/M/yyyy", "EventDate")

        # Collect and check types
        result = result_df.collect()

        # Assert that EventDate is converted to a Python date object
        self.assertIsInstance(result[0]['EventDate'], date)
        self.assertEqual(result[0]['EventDate'], date(2025, 11, 3))

        print("✅ convert_to_date_type() test passed!")


if __name__ == '__main__':
    unittest.main()
```
#### Example 2: 
- Here in this example I am going to simulate how to handle unstructured data in pyspark using a log_file from a server
- The data file is an apache webserver log file.
- We do have some pattern in this log file but this file is not even a semi-structure data file it is just a log dump
- So If I try to read this file into a dataFrame all I am going to get is a single row of strings
- I won't be getting columns in the dataFrame because the log file is an unstructured data file.
